In [1]:
import numpy as np
import pandas as pd
import os
import pickle


In [2]:
def main(path_save, file_stem):
    """
    Load peak analysis results (timing and amplitude) from a saved bundle.

    This function:
    1. Loads a pickle file containing previously computed peak statistics.
    2. Extracts key outputs including peak start indices, peak maxima,
       time-to-peak values, and peak amplitudes.
    3. Returns these values for downstream analysis or visualization.

    Parameters
    ----------
    path_save : str
        Directory where the peak results file is stored.
    file_stem : str
        Base filename used to locate the peak results file.

    Returns
    -------
    peaks_start : np.ndarray
        Array of peak start indices for each ROI.
    peaks_max : np.ndarray
        Array of peak maximum indices for each ROI.
    time_to_peak : np.ndarray
        Time difference (in seconds) between peak start and peak maximum.
    peaks_amp : np.ndarray
        Peak amplitudes (ΔF/F values).

    """

    file_data = os.path.join(path_save, f"{file_stem}_peaks.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    time_to_peak = bundle["time_to_peak"]
    peaks_start = bundle["peaks_start"]
    peaks_max = bundle["peaks_max"]
    peaks_amp = bundle["peaks_amp"]
    return peaks_start, peaks_max, time_to_peak, peaks_amp

 

In [3]:
data_source = {
"exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT":["pup1_spont","pup2_spont"]
}

In [5]:
path_dist = "Data"
genotype=pd.read_excel(os.path.join(path_dist, "genotypes_exp1_12.xlsx"), index_col=0)
fs=10 #Hz
nframes = 6000
# Inter-event interval statistics
pup_list=[]
IEI_mean=[]
IEI_sd=[]
n_comp=[]

for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        peaks_start, peaks_max, time_to_peak, peaks_amp = main(path_save,file_stem)

        #Calculate inter-peak interval
        interval=[]
        for peaks in peaks_max:
            mx=peaks[np.isfinite(peaks)]/fs
            int=mx[1:]-mx[:-1]
            interval.append(np.median(int))       
        pup_list.append(pup)
        IEI_mean.append(np.mean(interval))
        IEI_sd.append(np.std(interval))
        n_comp.append(len(interval))        
        print(f"Processing {file_stem} from {folder_exp}")

out=pd.DataFrame(index=pup_list, data={"IEI_mean": IEI_mean, "IEI_sd": IEI_sd,"n_comp":n_comp})
out["genotype"]=genotype["genotype"]
os.makedirs(os.path.join(path_dist,"NMF_analysis_results"), exist_ok=True)
out.to_excel(os.path.join(path_dist, "NMF_analysis_results", "nmf_compounds_inter_event_interval_statistics.xlsx"))



Processing pup1_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
Processing pup2_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT


In [6]:
# Frequency statistics
pup_list=[]
freq_mean=[]
freq_sd=[]
n_comp=[]
recording_length=nframes/fs # recording length in s
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        peaks_start, peaks_max, time_to_peak, peaks_amp = main(path_save,file_stem)

        #Calculate inter-peak interval
        frequency=[]
        for peaks in peaks_max:
            mx=peaks[np.isfinite(peaks)]
            npeak=len(mx)
            frequency.append(npeak/recording_length) # in Hz   
        pup_list.append(pup)
        freq_mean.append(np.mean(frequency))
        freq_sd.append(np.std(frequency))
        n_comp.append(len(frequency))        
        print(f"Processing {file_stem} from {folder_exp}")

out=pd.DataFrame(index=pup_list, data={"freq_mean": freq_mean, "freq_sd": freq_sd,"n_comp":n_comp})
out["genotype"]=genotype["genotype"]
os.makedirs(os.path.join(path_dist,"NMF_analysis_results"), exist_ok=True)
out.to_excel(os.path.join(path_dist, "NMF_analysis_results","nmf_compounds_peak_frequency_statistics.xlsx"))

Processing pup1_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
Processing pup2_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT


In [7]:
# Amplitude statistics
pup_list=[]
amp_mean=[]
amp_sd=[]
n_comp=[]
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        peaks_start, peaks_max, time_to_peak, peaks_amp = main(path_save,file_stem)

        #Calculate inter-peak interval
        amplitude=[]
        for peaks in peaks_amp:
            amp=peaks[np.isfinite(peaks)]
            npeak=len(amp)
            amplitude.append(np.mean(amp)) # in % 
        pup_list.append(pup)
        amp_mean.append(np.mean(amplitude))
        amp_sd.append(np.std(amplitude))
        n_comp.append(len(amplitude))        
        print(f"Processing {file_stem} from {folder_exp}")

out=pd.DataFrame(index=pup_list, data={"amp_mean": amp_mean, "amp_sd": amp_sd,"n_comp":n_comp})
out["genotype"]=genotype["genotype"]
os.makedirs(os.path.join(path_dist,"NMF_analysis_results"), exist_ok=True)
out.to_excel(os.path.join(path_dist,"NMF_analysis_results", "nmf_compounds_peak_amplitude_statistics.xlsx"))

Processing pup1_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
Processing pup2_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT


In [8]:
# Time to peak statistics
pup_list=[]
time_mean=[]
time_sd=[]
n_comp=[]
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        peaks_start, peaks_max, time_to_peak, peaks_amp = main(path_save,file_stem)

        #Calculate inter-peak interval
        time=[]
        for peaks in time_to_peak:
            t=peaks[np.isfinite(peaks)]
            time.append(np.median(t)) 
        pup_list.append(pup)
        time_mean.append(np.mean(time))
        time_sd.append(np.std(time))
        n_comp.append(len(time))        
        print(f"Processing {file_stem} from {folder_exp}")

out=pd.DataFrame(index=pup_list, data={"t2p_mean": time_mean, "t2p_sd": time_sd,"n_comp":n_comp})
out["genotype"]=genotype["genotype"]
os.makedirs(os.path.join(path_dist,"NMF_analysis_results"), exist_ok=True)
out.to_excel(os.path.join(path_dist, "NMF_analysis_results","nmf_compounds_time_to_peak_statistics.xlsx"))

Processing pup1_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
Processing pup2_spont from exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT
